In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import time
from collections import defaultdict
from datetime import datetime
import os
import json

In [2]:
def load_npy_dataset(file_path):
    """
    Load and prepare numpy dataset
    Returns X_train, y_train, X_test, y_test
    """
    data = np.load(file_path, allow_pickle=True).item()

    X_train = data["train"]["X"]
    X_train = data["train"]["X"].reshape(X_train.shape[0], -1)
    y_train = np.array([int(x) for x in data["train"]["y"]])
    X_test = data["test"]["X"]
    X_test = data["test"]["X"].reshape(X_test.shape[0], -1)
    y_test = np.array([int(x) for x in data["test"]["y"]])

    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

    return X_train, y_train, X_test, y_test

In [3]:
def prepare_data(X_train, X_test):
    """
    Prepare data by reshaping and scaling
    """
    start_time = time.time()

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    preprocessing_time = time.time() - start_time

    return X_train_scaled, X_test_scaled, preprocessing_time

In [4]:
def get_model_size(model):
    """
    Estimate the size of the model in bytes
    """
    import sys
    import pickle

    return sys.getsizeof(pickle.dumps(model))


def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Train and evaluate a model, return detailed metrics including timing
    """
    # Measure fit time
    start_fit_time = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start_fit_time

    # Measure prediction time for training data
    start_pred_train_time = time.time()
    y_pred_train = model.predict(X_train)
    pred_train_time = time.time() - start_pred_train_time

    # Measure prediction time for test data
    start_pred_test_time = time.time()
    y_pred_test = model.predict(X_test)
    pred_test_time = time.time() - start_pred_test_time

    # Calculate performance metrics
    train_accuracy = accuracy_score(y_train, y_pred_train)
    test_accuracy = accuracy_score(y_test, y_pred_test)

    # Calculate samples per second
    train_samples_per_second = X_train.shape[0] / pred_train_time
    test_samples_per_second = X_test.shape[0] / pred_test_time

    # Get classification report
    class_report = classification_report(y_test, y_pred_test)

    # Store all metrics in a dictionary
    metrics = {
        "model_name": model_name,
        "train_accuracy": float(train_accuracy),
        "test_accuracy": float(test_accuracy),
        "fit_time": float(fit_time),
        "pred_train_time": float(pred_train_time),
        "pred_test_time": float(pred_test_time),
        "train_samples_per_second": float(train_samples_per_second),
        "test_samples_per_second": float(test_samples_per_second),
        "classification_report": class_report,
        "model_size_bytes": get_model_size(model),
    }

    return metrics

In [5]:
def save_results(results, output_dir="results"):
    """
    Save results in both JSON and TXT formats
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_filename = os.path.join(output_dir, f"classification_results_{timestamp}")

    # Save as JSON
    json_filename = f"{base_filename}.json"
    with open(json_filename, "w") as f:
        # Convert results to JSON-serializable format
        json_results = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "results": results,
        }
        json.dump(json_results, f, indent=4)

    # Save as TXT
    txt_filename = f"{base_filename}.txt"
    with open(txt_filename, "w") as f:
        f.write("Time Series Classification Results\n")
        f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 100 + "\n\n")

        for dataset_name, dataset_results in results.items():
            f.write(f"\nResults for dataset: {dataset_name}\n")
            f.write("-" * 50 + "\n")

            for result in dataset_results:
                f.write(f"\nModel: {result['model_name']}\n")
                f.write("=" * (len(result["model_name"]) + 7) + "\n")
                f.write(f"Training Accuracy: {result['train_accuracy']:.4f}\n")
                f.write(f"Test Accuracy: {result['test_accuracy']:.4f}\n")
                f.write(f"Fit Time: {result['fit_time']:.4f} seconds\n")
                f.write(
                    f"Training Prediction Time: {result['pred_train_time']:.4f} seconds\n"
                )
                f.write(
                    f"Test Prediction Time: {result['pred_test_time']:.4f} seconds\n"
                )
                f.write(
                    f"Training Samples/Second: {result['train_samples_per_second']:.2f}\n"
                )
                f.write(
                    f"Test Samples/Second: {result['test_samples_per_second']:.2f}\n"
                )
                f.write(f"Model Size: {result['model_size_bytes'] / 1024:.2f} KB\n")
                f.write("\nClassification Report:\n")
                f.write(result["classification_report"])
                f.write("\n" + "-" * 50 + "\n")

    print(f"\nResults have been saved to:")
    print(f"JSON: {json_filename}")
    print(f"TXT: {txt_filename}")

    return json_filename, txt_filename

In [6]:
def run_classification(dataset_paths):
    """
    Run classification on multiple datasets with different models
    """
    models = {
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "SVM": SVC(kernel="rbf", random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=100, random_state=42
        ),
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Ridge Classifier": RidgeClassifier(random_state=42),
        "SGD Classifier": SGDClassifier(
            max_iter=1000, random_state=42, loss="modified_huber"
        ),
    }

    all_results = defaultdict(list)

    for dataset_path in dataset_paths:
        dataset_name = dataset_path.split("/")[-1].split(".")[0]
        print(f"\nProcessing dataset: {dataset_name}")

        # Load and prepare data
        X_train, y_train, X_test, y_test = load_npy_dataset(dataset_path)
        X_train_scaled, X_test_scaled, preprocessing_time = prepare_data(
            X_train, X_test
        )

        print(f"Preprocessing time: {preprocessing_time:.2f} seconds")

        dataset_results = []

        for model_name, model in models.items():
            print(f"\nTraining {model_name}...")
            result = evaluate_model(
                model, X_train_scaled, X_test_scaled, y_train, y_test, model_name
            )
            dataset_results.append(result)

            # Print results to console
            print(f"Model: {model_name}")
            print(f"Training Accuracy: {result['train_accuracy']:.4f}")
            print(f"Test Accuracy: {result['test_accuracy']:.4f}")
            print(f"Fit Time: {result['fit_time']:.4f} seconds")
            print("-" * 50)

        all_results[dataset_name] = dataset_results

    return all_results

In [7]:
# Example dataset paths - replace with your actual paths
dataset_paths = [
    "../data/raw/CounterMovementJump.npy",
    "../data/raw/MP_centred.npy",
    "../data/raw/MP50.npy",
    "../data/raw/synth_2lines.npy",
]

# Run classification
results = run_classification(dataset_paths)

# Save results
json_file, txt_file = save_results(results)

print("\nClassification completed successfully!")
print(f"Results saved to:\n{json_file}\n{txt_file}")


Processing dataset: CounterMovementJump
X_train shape: (419, 1152)
X_test shape: (179, 1152)
y_train shape: (419,)
y_test shape: (179,)
Preprocessing time: 0.00 seconds

Training Random Forest...
Model: Random Forest
Training Accuracy: 1.0000
Test Accuracy: 0.9330
Fit Time: 0.4136 seconds
--------------------------------------------------

Training SVM...
Model: SVM
Training Accuracy: 0.9857
Test Accuracy: 0.8939
Fit Time: 0.0381 seconds
--------------------------------------------------

Training KNN...
Model: KNN
Training Accuracy: 0.8353
Test Accuracy: 0.6201
Fit Time: 0.0004 seconds
--------------------------------------------------

Training Gradient Boosting...
Model: Gradient Boosting
Training Accuracy: 1.0000
Test Accuracy: 0.9385
Fit Time: 30.3684 seconds
--------------------------------------------------

Training Logistic Regression...
Model: Logistic Regression
Training Accuracy: 1.0000
Test Accuracy: 0.6872
Fit Time: 0.1629 seconds
----------------------------------------